In [12]:
'''
prompt:
Escreva codigo python que le todos os arquivos CSV disponiveis na pasta data/model-runs. 
O nome de cada arquivo CSV está no formato gemini-2.5-flash_answers.csv, em que tudo que vem antes de  "_answers.csv" é o nome do modelo. 

Leia cada arquivo CSV com o pandas. 
Cada linha do arquivo é uma questão da OAB que o modelo resolveu. 
Calcule o número de respostas certas comparando se "alternativa" == "correct". 
Crie um df em que cada row é um modelo, com a nota total e a nota percentual.

'''

import pandas as pd
import os
import glob
import re

# Caminho da pasta com os arquivos CSV
folder_path = '../data/processed/model-runs/'
csv_files = glob.glob(os.path.join(folder_path, '*_answers_irac*.csv'))

# Lista para guardar os resultados de cada modelo
model_results = []

for filepath in csv_files:
    # Extrai o nome do modelo a partir do nome do arquivo
    filename = os.path.basename(filepath)
    match = re.match(r'(.+)_answers_irac-(.*)\.csv$', filename)
    if match:
        model_name = match.group(1)
        irac = match.group(2)


    # Lê o CSV com pandas
    df = pd.read_csv(filepath)

    # Garante que as colunas esperadas estão presentes
    if 'alternativa' not in df.columns or 'correct' not in df.columns:
        print(f"Aviso: arquivo {filename} não contém as colunas esperadas.")
        continue

    # Conta quantas respostas estão corretas
    total_questions = len(df)
    correct_answers = (df['alternativa'] == df['correct']).sum()
    percentage = correct_answers / total_questions * 100

    # Adiciona os resultados à lista
    model_results.append({
        'modelo': model_name,
        'irac': irac,
        'questões': total_questions,
        'acertos': correct_answers,
        'percentual': round(percentage, 2)
    })

# Cria um DataFrame com os resultados
results_df = pd.DataFrame(model_results)

# Ordena do maior para o menor percentual de acertos
results_df = results_df.sort_values(by='percentual', ascending=False)

results_df.head(20)


,modelo,irac,questões,acertos,percentual
0,gemini-1.5-flash-8b,____,20,17,85.0
1,gemini-2.5-flash-lite-preview-06-17,____,20,17,85.0


In [13]:
import pandas as pd
import os
import glob
import re

# Caminho da pasta com os arquivos CSV
folder_path = '../data/processed/model-runs/'
csv_files = glob.glob(os.path.join(folder_path, '*_answers_irac*.csv'))

# Lista para guardar todos os dados juntos
all_data = []

for filepath in csv_files:
    # Extrai o nome do modelo a partir do nome do arquivo
    filename = os.path.basename(filepath)
    match = re.match(r'(.+)_answers_irac-(.*)\.csv$', filename)
    if match:
        model_name = match.group(1)
        irac = match.group(2)

    # Lê o CSV com pandas
    df = pd.read_csv(filepath)

    # Garante que as colunas esperadas estão presentes
    if not {'alternativa', 'correct', 'question_id'}.issubset(df.columns):
        print(f"Aviso: arquivo {filename} não contém as colunas esperadas.")
        continue

    # Adiciona colunas de metadados
    df['modelo'] = model_name
    df['irac'] = irac
    df['acertou'] = df['alternativa'] == df['correct']

    # Armazena os dados
    all_data.append(df)

# Junta todos os dados em um único DataFrame
full_df = pd.concat(all_data, ignore_index=True)

# Calcula a acurácia por question_id
accuracy_by_question = (
    full_df.groupby('question_id')['acertou']
    .mean()
    .reset_index()
    .rename(columns={'acertou': 'acurácia'})
    .sort_values(by='acurácia', ascending=False)
)

# Exibe as 20 questões com maior acurácia
accuracy_by_question.head(20)


,question_id,acurácia
0,oab-101.pdf-028,1.0
8,oab-22.pdf-221,1.0
18,oab-72.pdf-260,1.0
14,oab-299.pdf-023,1.0
13,oab-270.pdf-249,1.0
12,oab-27.pdf-268,1.0
11,oab-258.pdf-124,1.0
10,oab-246.pdf-006,1.0
7,oab-179.pdf-270,1.0
6,oab-171.pdf-191,1.0
